#### Setup

Step 0: Install (alpha version of) the library

In [1]:
!pip install model-signing==0.0.2a0


[notice] A new release of pip is available: 23.2.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


#### Obtain some ML model

In general, here we'd train a model and then sign it. For the demo, we would instead just download a model from the internet and use that.

In [2]:
!rm -rf granite-3b-code-base-2k/

In [3]:
!git clone https://huggingface.co/ibm-granite/granite-3b-code-base-2k

Cloning into 'granite-3b-code-base-2k'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 132 (delta 69), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (132/132), 660.69 KiB | 5.60 MiB/s, done.
Resolving deltas: 100% (69/69), done.
Filtering content: 100% (2/2), 2.48 GiB | 37.88 MiB/s, done.


In [4]:
!rm -rf granite-3b-code-base-2k/.git
!ls -lh granite-3b-code-base-2k/

total 6.5G
-rw-r--r--. 1 ifont ifont  680 Sep 18 08:35 config.json
-rw-r--r--. 1 ifont ifont  137 Sep 18 08:35 generation_config.json
-rw-r--r--. 1 ifont ifont 4.7G Sep 18 08:37 model-00001-of-00002.safetensors
-rw-r--r--. 1 ifont ifont 1.9G Sep 18 08:36 model-00002-of-00002.safetensors
-rw-r--r--. 1 ifont ifont  41K Sep 18 08:35 model.safetensors.index.json
-rw-r--r--. 1 ifont ifont  11K Sep 18 08:35 README.md
-rw-r--r--. 1 ifont ifont 1020 Sep 18 08:35 special_tokens_map.json
-rw-r--r--. 1 ifont ifont 4.1K Sep 18 08:35 tokenizer_config.json
-rw-r--r--. 1 ifont ifont 2.0M Sep 18 08:35 tokenizer.json


In [5]:
!du -sh granite-3b-code-base-2k

6.5G	granite-3b-code-base-2k


#### Sign the model

In [6]:
import base64
import json
import pathlib

from model_signing.hashing import file
from model_signing.hashing import memory
from model_signing.serialization import serialize_by_file
from model_signing.serialization import serialize_by_file_shard
from model_signing.signing import in_toto
from model_signing.signing import sigstore

Hashing layer:

In [7]:
def file_hasher_factory(path: pathlib.Path) -> file.FileHasher:
    return file.SimpleFileHasher(path, memory.SHA256())

Serialization layer:

In [8]:
serializer = serialize_by_file.ManifestSerializer(file_hasher_factory, allow_symlinks=True)
manifest = serializer.serialize(pathlib.Path("granite-3b-code-base-2k"))

Signing layer:

In [9]:
payload = in_toto.DigestsIntotoPayload.from_manifest(manifest)
signer = sigstore.SigstoreDSSESigner(use_ambient_credentials=False, use_staging=True)
signature = signer.sign(payload)
signature.write(pathlib.Path("model.sig"))

Go to the following link in a browser:

	https://oauth2.sigstage.dev/auth/auth?response_type=code&client_id=sigstore&client_secret=&scope=openid+email&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&code_challenge=5YiQN4KnmCat_1VBZ_6sDmbTbnOnl8N6C5fpOrtQeOQ&code_challenge_method=S256&state=8886e3c6-ab67-4666-ae38-f531a5d74714&nonce=d6cd9c63-83cf-4222-8e9a-a23e6ae0bfdd


Enter verification code:  zvpvjm4ngvlxyxidzzcjmcoav


##### Inspect signature (optional)

In [10]:
!cat model.sig | jq .

{
  "mediaType": "application/vnd.dev.sigstore.bundle.v0.3+json",
  "verificationMaterial": {
    "certificate": {
      "rawBytes": "MIICyjCCAk+gAwIBAgIUAZ5rGIxXimh7zYKck1wkRQMTFD8wCgYIKoZIzj0EAwMwNzEVMBMGA1UEChMMc2lnc3RvcmUuZGV2MR4wHAYDVQQDExVzaWdzdG9yZS1pbnRlcm1lZGlhdGUwHhcNMjQwOTE4MTUzNzM0WhcNMjQwOTE4MTU0NzM0WjAAMFkwEwYHKoZIzj0CAQYIKoZIzj0DAQcDQgAECb2CHlKZSXOjG8ZGzjXi/Yx6iycLz09cBEmgtJh5zEOTsmxlk1UQcd0rR1hfe5a4AuZUE6EnsASPBcHFF5a4qKOCAW4wggFqMA4GA1UdDwEB/wQEAwIHgDATBgNVHSUEDDAKBggrBgEFBQcDAzAdBgNVHQ4EFgQU0n4gkIqnJd1keiYAirClIzwVSoswHwYDVR0jBBgwFoAUcYYwphR8Ym/599b0BRp/X//rb6wwHgYDVR0RAQH/BBQwEoEQaWZvbnRAcmVkaGF0LmNvbTApBgorBgEEAYO/MAEBBBtodHRwczovL2FjY291bnRzLmdvb2dsZS5jb20wKwYKKwYBBAGDvzABCAQdDBtodHRwczovL2FjY291bnRzLmdvb2dsZS5jb20wgYoGCisGAQQB1nkCBAIEfAR6AHgAdgArMLzcaIjJ4uHYJiledB9IOTGWAvKcM8teQ0D+sqyGegAAAZIFyCG+AAAEAwBHMEUCIH6AJttpIbHJKgWwRuZVwn28CZMp5/11RoTVZbuPbczvAiEAuJHPbLRZtk22QnzM+KI2Qe00KzPjcUZY4ZIaqtGd7gAwCgYIKoZIzj0EAwMDaQAwZgIxAIcJTqD9zvXo8En7BMFCbwpouSa1zVGo6jk3/rXoBn

In [11]:
!cat model.sig | jq .dsseEnvelope.payload -r | base64 -d | jq .

{
  "_type": "https://in-toto.io/Statement/v1",
  "subject": [
    {
      "name": ".gitattributes",
      "digest": {
        "sha256": "11ad7efa24975ee4b0c3c3a38ed18737f0658a5f75a0a96787b576a78a023361"
      },
      "annotations": {
        "actual_hash_algorithm": "file-sha256"
      }
    },
    {
      "name": "README.md",
      "digest": {
        "sha256": "fa166922cca62cd8acdd54d1964abe91d1b58cfb9afd6e4c599cb09912d9a9f9"
      },
      "annotations": {
        "actual_hash_algorithm": "file-sha256"
      }
    },
    {
      "name": "config.json",
      "digest": {
        "sha256": "bdc265c2e79f1c6b6c0f4f25313e0d1cc936e0fd4e1e082bb26f65a4bd0fa557"
      },
      "annotations": {
        "actual_hash_algorithm": "file-sha256"
      }
    },
    {
      "name": "generation_config.json",
      "digest": {
        "sha256": "e53a4633c99233a65025b869b1e0fc5747a737d39d57f6a40a78a64a570d3628"
      },
      "annotations": {
        "actual_hash_algorithm": "file-sha256"
      }
    

#### Verify the model

In [12]:
signature = sigstore.SigstoreSignature.read(pathlib.Path("model.sig"))
verifier = sigstore.SigstoreDSSEVerifier(
    identity="ifont@redhat.com",
    oidc_issuer="https://accounts.google.com",
    use_staging=True
)
signature_manifest = verifier.verify(signature)

Verify local model against signature payload

In [13]:
if signature_manifest == serializer.serialize(pathlib.Path("granite-3b-code-base-2k")):
  print("Signature is valid")
else:
  print("Signature is invalid")

Signature is valid


#### Alter the model and then verify it again

In [14]:
!rm granite-3b-code-base-2k/config.json

In [15]:
signature = sigstore.SigstoreSignature.read(pathlib.Path("model.sig"))
verifier = sigstore.SigstoreDSSEVerifier(
    identity="ifont@redhat.com",
    oidc_issuer="https://accounts.google.com",
    use_staging=True
)
signature_manifest = verifier.verify(signature)

In [16]:
if signature_manifest == serializer.serialize(pathlib.Path("granite-3b-code-base-2k")):
  print("Signature is valid")
else:
  print("Signature is invalid")

Signature is invalid


### Optional explorations

Serialize the model using a different hash algorithm, and a different serialization method (shard large files and hash each shard individually).

In [17]:
def sharded_file_hasher_factory(path: pathlib.Path, start: int, end: int) -> file.FileHasher:
    return file.ShardedFileHasher(path, memory.BLAKE2(), start=start, end=end)

In [18]:
serializer = serialize_by_file_shard.ManifestSerializer(sharded_file_hasher_factory, allow_symlinks=True)
manifest = serializer.serialize(pathlib.Path("granite-3b-code-base-2k"))

In [19]:
payload = in_toto.ShardDigestsIntotoPayload.from_manifest(manifest)
signer = sigstore.SigstoreDSSESigner(use_ambient_credentials=False, use_staging=True)
signature = signer.sign(payload)
signature.write(pathlib.Path("model.sig"))

Go to the following link in a browser:

	https://oauth2.sigstage.dev/auth/auth?response_type=code&client_id=sigstore&client_secret=&scope=openid+email&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&code_challenge=064YBgh60wZbSPbDl5EX1x5Om1ysWlN2RVSycebio2g&code_challenge_method=S256&state=308c3075-94ee-4159-97b6-87f4b43ddac4&nonce=f665bd57-b4ef-40b8-9c9b-2d80f97751e6


Enter verification code:  edpaq4354lqxrwmh7pwqiaoqt


In [20]:
!cat model.sig | jq .dsseEnvelope.payload -r | base64 -d | jq .

{
  "_type": "https://in-toto.io/Statement/v1",
  "subject": [
    {
      "name": ".gitattributes:0:1519",
      "digest": {
        "sha256": "a13c8cb6cf3541459ed1a18e631560df74fa2f7de6ba9d2337252eab40281d5be357594027d4a6dff4d3db9a50857d07bc9debce042cac12ac519cd76cdc7610"
      },
      "annotations": {
        "actual_hash_algorithm": "file-blake2b-1000000"
      }
    },
    {
      "name": "README.md:0:10325",
      "digest": {
        "sha256": "4ce34107ce28af7e659c957e4e569d07f0d46b21707a915e50e14bef13db584b2650b1497cc0c2ebb467493e725e8608a5f26d9ddc6c06435d28e164a4fa7908"
      },
      "annotations": {
        "actual_hash_algorithm": "file-blake2b-1000000"
      }
    },
    {
      "name": "generation_config.json:0:137",
      "digest": {
        "sha256": "09022e0824a47a5bc8c565fa2bcfdbe4dd051bf903ae49d478de1130d34b5f6746567c5de8da364e24a735b85be0c31db1f27b1141dd3e0050a93ce06fe1cf30"
      },
      "annotations": {
        "actual_hash_algorithm": "file-blake2b-1000000"
    

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [21]:
signature = sigstore.SigstoreSignature.read(pathlib.Path("model.sig"))
verifier = sigstore.SigstoreDSSEVerifier(
    identity="ifont@redhat.com",
    oidc_issuer="https://accounts.google.com",
    use_staging=True
)
signature_manifest = verifier.verify(signature)

In [22]:
if signature_manifest == serializer.serialize(pathlib.Path("granite-3b-code-base-2k")):
  print("Signature is valid")
else:
  print("Signature is invalid")

Signature is valid
